# D19 S2 A2 - Transfer Learning Sample Efficiency

Frozen PCA backbone with a kNN head versus kNN on raw high-dimensional features.


In [1]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split


def make_data(seed=0, n=2000, D=500, k=5, classes=4, csep=3.0, onoise=16.0):
    rng = np.random.default_rng(seed)
    centers = rng.normal(0, csep, size=(classes, k))
    y = rng.integers(0, classes, size=n)
    Z = centers[y] + rng.normal(0, 1.0, size=(n, k))
    W = rng.normal(0, 1.0, size=(k, D))
    X = Z @ W + rng.normal(0, onoise, size=(n, D))
    return X.astype(float), y


In [2]:
def fit_backbone(X_pool, n_components=5):
    backbone = PCA(n_components=n_components, random_state=0)
    backbone.fit(X_pool)
    return backbone


def extract(backbone, X):
    return backbone.transform(X)


def transfer_score(backbone, Xtr, ytr, Xte, yte, k_neighbors=5):
    Xtr_features = extract(backbone, Xtr)
    Xte_features = extract(backbone, Xte)
    model = KNeighborsClassifier(n_neighbors=k_neighbors)
    model.fit(Xtr_features, ytr)
    return round(float(model.score(Xte_features, yte)), 4)


def raw_score(Xtr, ytr, Xte, yte, k_neighbors=5):
    model = KNeighborsClassifier(n_neighbors=k_neighbors)
    model.fit(Xtr, ytr)
    return round(float(model.score(Xte, yte)), 4)


def sample_efficiency(backbone, Xtr, ytr, Xte, yte, sizes):
    results = {}
    for n in sizes:
        results[n] = {
            "transfer": transfer_score(backbone, Xtr[:n], ytr[:n], Xte, yte),
            "raw": raw_score(Xtr[:n], ytr[:n], Xte, yte),
        }
    return results


## Verification


In [3]:
X, y = make_data(0)
X_pool, X_test, y_pool, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=0
)

bb = fit_backbone(X_pool, n_components=5)
tbl = sample_efficiency(bb, X_pool, y_pool, X_test, y_test, [30, 60, 200])
for n, r in tbl.items():
    print(n, r["transfer"], r["raw"])


30 0.9867 0.8817
60 0.9867 0.8983
200 0.9867 0.9583


## Autograder-style checks


In [4]:
# 1-2: fitted 5D PCA backbone and extraction shape
assert hasattr(bb, "components_")
assert bb.n_components == 5
assert extract(bb, X_test[:10]).shape == (10, 5)

# 3-4: score types and ranges
ts = transfer_score(bb, X_pool[:60], y_pool[:60], X_test, y_test)
rs = raw_score(X_pool[:60], y_pool[:60], X_test, y_test)
assert isinstance(ts, float) and 0 < ts <= 1
assert isinstance(rs, float) and 0 < rs <= 1

# 5: expected table structure
assert set(tbl.keys()) == {30, 60, 200}
for n in tbl:
    assert set(tbl[n].keys()) == {"transfer", "raw"}

# 6: more labels do not hurt frozen-feature accuracy for this seeded dataset
assert tbl[200]["transfer"] >= tbl[30]["transfer"]

# 7: clear transfer-learning margin at 60 labels
assert tbl[60]["transfer"] - tbl[60]["raw"] > 0.02

# 8: strong frozen-feature accuracy at 60 labels
assert tbl[60]["transfer"] > 0.85

print("All 8 autograder-style checks passed.")


All 8 autograder-style checks passed.
